# Samaritan — serve a local GGUF from Colab

Puts the model on a rented GPU and leaves your machine to do nothing but
issue HTTP calls. `reason_eval` is ~8 MB resident, so a 4 GB laptop GPU stops
being the bottleneck for an overnight eval.

**Runtime → Change runtime type → L4, then Run all.** The last cell prints the
`SAMARITAN_URL` to paste locally.

### Matching the local serving config exactly

The harness pins `temperature`, `max_tokens`, `repeat_penalty` and `seed` in
every request body, so those travel with the client and are already identical.
Two things do **not** travel and must be set on the model itself:

- **`num_ctx 8192`** — Ollama silently *context-shifts* past its window: it
  drops the oldest tokens and returns a normal-looking reply with no error. At
  the 2048 default a 6000-token budget would quietly truncate every long
  answer, and the failures would look like wrong answers rather than clipping.
- **`repeat_penalty 1.0`** — the harness sends this, but `repeat_penalty` is an
  Ollama-native option, not an OpenAI one, so the `/v1` endpoint ignores it and
  would otherwise apply Ollama's own 1.1 default.

Both are set in the Modelfile below. Without them this is not the same
experiment as the local run, however similar the numbers look.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 1. Get the GGUF onto the runtime

Drive is the sane route for 2.4 GB — upload once from your machine, reuse it
every session. `files.upload()` pushes through the browser tab and tends to
die partway at this size.

Put the file anywhere in Drive and set `GGUF_NAME` to its filename.

In [ ]:
GGUF_NAME = 'samaritan-student-v1-185trace-q4_k_m.gguf'  # <-- the file in your Drive

import os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')

hits = glob.glob(f'/content/drive/MyDrive/**/{GGUF_NAME}', recursive=True)
assert hits, (f'{GGUF_NAME} not found in Drive. Upload it to MyDrive '
              '(any folder) and re-run this cell.')
SRC = hits[0]
print('found:', SRC, f'({os.path.getsize(SRC)/1e9:.2f} GB)')

# Copy to local disk: serving straight off the Drive FUSE mount is slow and
# drops out on long runs.
GGUF = '/content/model.gguf'
if not os.path.exists(GGUF) or os.path.getsize(GGUF) != os.path.getsize(SRC):
    shutil.copy(SRC, GGUF)
print('local copy:', GGUF, f'({os.path.getsize(GGUF)/1e9:.2f} GB)')

## 2. Ollama

In [ ]:
import os, subprocess, time, urllib.request, json, shutil, glob
!curl -fsSL https://ollama.com/install.sh | sh
assert shutil.which('ollama'), 'ollama install failed (see output above)'

subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
time.sleep(2)
env = {**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '-1',
       'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0'}
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'),
                       stderr=subprocess.STDOUT, env=env)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=5)
        print('daemon up'); break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit('daemon did not start:\n' + open('ollama.log').read()[-2000:])

## 3. Register the model under the alias the harness expects

`samaritan-playout` is what `SAMARITAN_MODEL` defaults to, so the same local
command works against local or remote serving with only the URL changed.

In [ ]:
CTX = 8192          # must match the local run, and exceed MAX_TOKENS + prompt
REPEAT_PENALTY = 1.0  # harness default; /v1 ignores the request field

with open('Modelfile', 'w') as f:
    f.write(f'FROM {GGUF}\n'
            f'PARAMETER num_ctx {CTX}\n'
            f'PARAMETER repeat_penalty {REPEAT_PENALTY}\n')
print(open('Modelfile').read())
subprocess.run(['ollama', 'create', 'samaritan-playout', '-f', 'Modelfile'], check=True)

lst = subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout
assert 'samaritan-playout' in lst, 'alias not created:\n' + lst
print(lst)

print('warming (first load 30-60s)...')
body = json.dumps({'model': 'samaritan-playout',
    'messages': [{'role': 'user', 'content': 'What is 2+2? Reply with just the number.'}],
    'stream': False, 'options': {'num_predict': 64}}).encode()
req = urllib.request.Request('http://127.0.0.1:11434/api/chat', data=body,
    headers={'Content-Type': 'application/json'})
r = json.loads(urllib.request.urlopen(req, timeout=600).read())
print('[OK] generates:', r['message']['content'][:80].replace(chr(10), ' '))

### Confirm the context actually took

Worth thirty seconds: a `num_ctx` that silently stayed at 2048 produces a run
that looks fine and measures nothing, which is the expensive kind of wrong.

In [ ]:
log = open('ollama.log').read()
import re
hits = re.findall(r'n_ctx\s*=\s*(\d+)', log)
print('n_ctx seen in the server log:', hits[-5:] if hits else '(none logged yet)')
if hits:
    got = max(int(h) for h in hits)
    print(f'{"OK" if got >= CTX else "MISMATCH"}: server built a {got}-token context '
          f'(asked for {CTX})')
    assert got >= CTX, 'context smaller than requested - fix before evaluating'

## 4. Public tunnel

In [ ]:
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
try:
    cf.terminate()
except Exception:
    pass
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434',
                       '--http-host-header', 'localhost:11434'],
                      stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
public = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    if m:
        public = m.group(0); break
assert public, 'no tunnel URL - cf.log:\n' + open('cf.log').read()[-1000:]

def _get(url, timeout=60):
    req = urllib.request.Request(url, headers={'Authorization': 'Bearer ollama'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.read().decode()

ok = False
for _ in range(8):
    try:
        s, b = _get(public + '/v1/models')
        if s == 200 and 'samaritan-playout' in b:
            ok = True; break
    except Exception:
        pass
    time.sleep(5)
assert ok, f'tunnel up at {public} but /v1/models did not serve the alias'

print('\n' + '=' * 62)
print('Run this locally (PowerShell):')
print()
print(f'  $env:SAMARITAN_URL = "{public}/v1"')
print('=' * 62)

## 5. Keep-alive

Leave this running while you evaluate. Stop it when done — the runtime bills
for as long as it is allocated, whether or not it is busy.

In [ ]:
from datetime import datetime
UNITS_PER_HOUR = 5.3
MAX_HOURS = 8          # 0 = no cap

start = time.time()
beat = 0
while True:
    beat += 1
    hrs = (time.time() - start) / 3600
    if MAX_HOURS and hrs > MAX_HOURS:
        print(f'\nMAX_HOURS ({MAX_HOURS}) reached - stopping keep-alive.')
        break
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,utilization.gpu',
                          '--format=csv,noheader'], capture_output=True,
                         text=True).stdout.strip()
    alive = 'up' if cf.poll() is None else 'DOWN'
    print(f'[{datetime.now():%H:%M:%S}] beat {beat:>4} | gpu {gpu} | tunnel {alive} '
          f'| {hrs:.2f} h | ~{hrs * UNITS_PER_HOUR:.1f} units', flush=True)
    time.sleep(60)